# NBA Quant AI — Colab B (TabICL v2)

TabICL in-context learning, **186-feature** version matching iter-129 setup that produced Colab's current Brier record 0.21514.

**Strategy** (distinct from Colab A)
- TabICL (not TabPFN) — in-context learning with demonstration retrieval
- Mutation cap 0.12 (more flexible than TabPFN's 0.08)
- Random seed **1337** (not 42) — different trajectory than Colab A
- CPCV 10-fold (not 80/20 temporal) — combinatorial purged cross-validation
- Target: Brier < 0.210 stretch

**Writes** `data/departments/gpu-results-colab-b.jsonl`.

**Cell structure**: class-based runner — `ColabBRunner.run()` is the single entrypoint.

In [ ]:
# Cell 1 — Install deps
!pip install -q tabicl pandas numpy scikit-learn pyarrow

In [ ]:
# Cell 2 — Drive + repo mount
from google.colab import drive  # type: ignore[import-not-found]
drive.mount('/content/drive')

import os, subprocess
REPO_DIR = '/content/mon-ipad'
REPO_URL = 'https://github.com/LBJLincoln/mon-ipad.git'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

In [ ]:
# Cell 3 — Class-based runner (config + logic bundled)
from __future__ import annotations

import itertools
import json
import logging
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import numpy as np  # type: ignore[import-not-found]
import pandas as pd  # type: ignore[import-not-found]
from sklearn.metrics import brier_score_loss  # type: ignore[import-not-found]


@dataclass(frozen=True)
class ColabBConfig:
    repo_root: Path = Path('/content/mon-ipad')
    feature_matrix: Path = Path('/content/mon-ipad/data/nba-agent/feature-matrix.parquet')
    fallback_csv: Path = Path('/content/mon-ipad/data/nba-agent/feature-matrix.csv')
    n_features: int = 186
    random_state: int = 1337
    mutation_cap: float = 0.12
    cpcv_n_folds: int = 10
    cpcv_embargo_pct: float = 0.02
    target_brier: float = 0.210
    results_path: Path = Path('/content/mon-ipad/data/departments/gpu-results-colab-b.jsonl')
    run_tag: str = 'colab-b-tabicl-v2'
    sweep_ctx_sizes: tuple[int, ...] = (512, 1024, 2048)
    sweep_temps: tuple[float, ...] = (0.95, 1.0, 1.08)


class ColabBRunner:
    def __init__(self, cfg: ColabBConfig):
        self.cfg = cfg
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s [%(levelname)s] %(message)s',
        )
        self.log = logging.getLogger('colab-b')

    def load(self) -> pd.DataFrame:
        if self.cfg.feature_matrix.exists():
            df = pd.read_parquet(self.cfg.feature_matrix)
        elif self.cfg.fallback_csv.exists():
            df = pd.read_csv(self.cfg.fallback_csv)
        else:
            raise FileNotFoundError(self.cfg.feature_matrix)
        if 'game_date' in df.columns:
            df = df.sort_values('game_date').reset_index(drop=True)
        self.log.info('loaded shape=%s', df.shape)
        return df

    def select_features(self, df: pd.DataFrame, label_col: str = 'y_home_win') -> tuple[pd.DataFrame, np.ndarray, list[str]]:
        drop = {label_col, 'game_id', 'game_date', 'home', 'away'}
        numeric = df.select_dtypes(include=[np.number]).drop(columns=[c for c in drop if c in df.columns], errors='ignore')
        variance = numeric.var().sort_values(ascending=False)
        chosen = variance.head(self.cfg.n_features).index.tolist()
        X = numeric[chosen].fillna(0.0)
        y = df[label_col].astype(int).values
        self.log.info('selected %d features (variance-ranked)', len(chosen))
        return X, y, chosen

    def cpcv_folds(self, n_rows: int) -> Iterable[tuple[np.ndarray, np.ndarray]]:
        n_folds = self.cfg.cpcv_n_folds
        fold_size = n_rows // n_folds
        embargo = max(1, int(n_rows * self.cfg.cpcv_embargo_pct))
        for k in range(n_folds):
            test_lo = k * fold_size
            test_hi = test_lo + fold_size if k < n_folds - 1 else n_rows
            train_mask = np.ones(n_rows, dtype=bool)
            train_mask[max(0, test_lo - embargo):min(n_rows, test_hi + embargo)] = False
            test_mask = np.zeros(n_rows, dtype=bool)
            test_mask[test_lo:test_hi] = True
            yield np.where(train_mask)[0], np.where(test_mask)[0]

    def eval_params(self, X: pd.DataFrame, y: np.ndarray, ctx: int, temp: float) -> float:
        from tabicl import TabICLClassifier  # type: ignore[import-not-found]
        scores: list[float] = []
        for tr_ix, te_ix in self.cpcv_folds(len(X)):
            m = TabICLClassifier(
                n_estimators=1,
                softmax_temperature=temp,
                random_state=self.cfg.random_state,
            )
            sub_tr = tr_ix[-ctx:]
            m.fit(X.iloc[sub_tr].values, y[sub_tr])
            p = m.predict_proba(X.iloc[te_ix].values)[:, 1]
            scores.append(float(brier_score_loss(y[te_ix], p)))
        return float(np.mean(scores))

    def sweep(self, X: pd.DataFrame, y: np.ndarray) -> list[dict[str, Any]]:
        results: list[dict[str, Any]] = []
        for ctx, temp in itertools.product(self.cfg.sweep_ctx_sizes, self.cfg.sweep_temps):
            brier = self.eval_params(X, y, ctx, temp)
            self.log.info('ctx=%d temp=%.2f brier_cv=%.5f', ctx, temp, brier)
            results.append({'ctx': ctx, 'temp': temp, 'brier_cv_mean': brier})
        return results

    def persist(self, sweeps: list[dict[str, Any]], X: pd.DataFrame) -> dict[str, Any]:
        best = min(sweeps, key=lambda s: s['brier_cv_mean'])
        self.cfg.results_path.parent.mkdir(parents=True, exist_ok=True)
        record = {
            'ts': datetime.now(timezone.utc).isoformat(timespec='seconds'),
            'dept': 'gpu-colab-b',
            'notebook': 'nba_tabicl_v2.ipynb',
            'run_tag': self.cfg.run_tag,
            'mutation_cap': self.cfg.mutation_cap,
            'random_state': self.cfg.random_state,
            'cpcv_n_folds': self.cfg.cpcv_n_folds,
            'target_brier': self.cfg.target_brier,
            'n_rows': int(len(X)),
            'n_features': self.cfg.n_features,
            'brier_best': best['brier_cv_mean'],
            'best_params': {k: v for k, v in best.items() if k != 'brier_cv_mean'},
            'sweeps': sweeps,
            'hit_target': best['brier_cv_mean'] < self.cfg.target_brier,
        }
        with self.cfg.results_path.open('a') as fh:
            fh.write(json.dumps(record) + '\n')
        self.log.info('wrote record to %s', self.cfg.results_path)
        return record

    def run(self) -> dict[str, Any]:
        df = self.load()
        X, y, _ = self.select_features(df)
        sweeps = self.sweep(X, y)
        return self.persist(sweeps, X)


In [ ]:
# Cell 4 — Run it
record = ColabBRunner(ColabBConfig()).run()
print(json.dumps(record, indent=2))

In [ ]:
# Cell 5 — Auto-commit via safe_commit.sh (optional)
import subprocess
cfg = ColabBConfig()
rel = str(cfg.results_path.relative_to(cfg.repo_root))
r = subprocess.run(
    ['bash', 'scripts/lib/safe_commit.sh', 'COLAB_B',
     f"colab-b TabICL brier_best={record['brier_best']:.5f}", rel],
    cwd=cfg.repo_root, capture_output=True, text=True,
)
print('stdout:', r.stdout[-800:])
print('stderr:', r.stderr[-800:])
print('returncode:', r.returncode)